# AMB kernel · parameter / system sanity

학습 전에 physical parameter, kernel, solver, stationarity 및 EPR 추정량을 확인하는 별도 notebook입니다. 선택한 run의 profile 영상을 여기서 재생할 수 있으며 NN 학습은 실행하지 않습니다. 학습과 NN/physical EP 영상, time-averaged EP map, shell spectrum은 [Corr_AMB_Kernel_ShellForce.ipynb](Corr_AMB_Kernel_ShellForce.ipynb)에서 실행합니다.

기본 선택은 **M01 / primary / seed 73031 한 run**, grid는 **64×64**입니다. `dt=0.01`, burn-in 10,000 step, measurement 1,000 step과 circle 초기 profile을 사용합니다. 여러 control·seed·dt는 아래에서 직접 선택한 경우에만 실행됩니다. 실행 완료가 stationary EPR 검증을 의미하지 않으며, 각 진단의 표본 수와 수렴 여부를 함께 확인합니다.

GPU가 있으면 같은 physical parameter와 dt의 독립 seed들을 batch로 함께 적분하고, CPU에서는 기준 SciPy simulator를 사용합니다. kernel의 전체 periodic support와 signed channel/tail 기여를 보존합니다.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone, timedelta
from time import perf_counter
import json
import sys

HERE = Path.cwd().resolve()
ROOT = next((path for path in [HERE, *HERE.parents, HERE / 'CNEEP_v2']
             if (path / 'models/NEEP_ShellForce_2D.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter in CNEEP_v2, its notebooks folder, or its parent.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from tqdm.auto import tqdm
from data.amb_kernel.model import AMBKernelModel, MODEL_NAMES, pilot_config, control_configs
from data.amb_kernel.experiment import RunConfig, run_experiment, load_experiment
from data.amb_kernel.dataset import resolve_simulation_device
from data.amb_kernel.diagnostics import compare_epr_methods
from data.amb_kernel.plotting import render_profile_video

STARTED = perf_counter()
SHOW_PROGRESS = True
BAR_FORMAT = '{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed} elapsed, ETA {remaining}, {rate_fmt}{postfix}]'

def close_fields(result):
    if isinstance(result.fields, np.memmap):
        result.fields._mmap.close()

print('Repository:', ROOT)

## 1. 선택할 모델·parameter·control·seed·dt

`SELECTED_MODELS`, `CASE_NAMES`, `SEEDS`가 실제 실행 범위를 정합니다. `MODEL_OVERRIDES`는 모델별 물리 parameter를 바꿉니다. 가능한 control 이름을 아래에서 확인한 뒤 `CASE_NAMES`에 추가합니다. 예를 들어 M01의 `('primary', 'reciprocal', 'no_source')`, M03의 `('primary', 'no_flow', 'reflected')`를 비교할 수 있습니다.

`DT_FACTORS=(1.0, 0.5, 0.25)`는 dt/2·dt/4 비교입니다. 이때 burn-in/measurement/block 시간과 저장 observation 간격을 **물리 시간 기준으로 동일하게** 맞춥니다. 정수 step으로 표현되지 않는 조합은 오류로 알립니다. Single seed는 independent-seed 불확실성을 추정할 수 없으므로 실제 검증 시 `SEEDS`를 늘리세요.

`DT` 자체만 줄이면 같은 step 수의 물리 시간도 줄어듭니다. 예를 들어 DT=0.001에서 측정 1,000 step은 시간 1입니다. 시간 간격 오차를 비교할 때는 기준 DT를 유지하고 DT_FACTORS를 사용하세요. Block SE는 block 독립성이 확인되기 전까지 잠정값입니다.

In [ ]:
SELECTED_MODELS = ('M01',)
CASE_NAMES = {model_id: ('primary',) for model_id in MODEL_NAMES}
MODEL_OVERRIDES = {model_id: {} for model_id in MODEL_NAMES}
# Example: MODEL_OVERRIDES['M01'] = dict(g2=0.2, phi_bar=0.3)
SEEDS = (73031,)
N = 64
DT = 0.01
BURN_IN_STEPS = 10000
MEASUREMENT_STEPS = 1000
BLOCK_STEPS = 100
OBSERVATION_STRIDE = 1
OBSERVATION_LAGS = (1, 4, 16)  # base-dt integration steps
INITIAL_MODE = 'circle'
INITIAL_NOISE_SCALE = 0.0
CIRCLE_RADIUS_FRACTION = 0.25
CIRCLE_WIDTH = 1.0
CIRCLE_AMPLITUDE = 1.0
DT_FACTORS = (1.0,)  # select (1.0, 0.5, 0.25) for a matched-physical-time comparison
MAX_DISTANCE = 3
SIMULATION_DEVICE = 'auto'  # 'auto', 'cpu', 'cuda', or 'cuda:0'
SIMULATION_BATCH_SIZE = 32
RUN_TO_PLOT = None  # None selects the first run; set a run_id after inspecting the plan
VIDEO_RUN_PATH = None  # existing runs/<run_id> folder; None uses RUN_TO_PLOT / first run
VIDEO_FRAME_STRIDE = 5
VIDEO_MAX_FRAMES = 200
VIDEO_FPS = 10

# Optional M00 circular-droplet candidate: uncomment this entire block together.
# SELECTED_MODELS = ('M00',)
# MODEL_OVERRIDES['M00'] = dict(a=-1.0, b=1.0, kappa=2.0, g1=0.1,
#                              lambda0=0.05, D=0.01, phi_bar=-0.57, ell=1.2)
# INITIAL_MODE = 'circle'
# INITIAL_NOISE_SCALE = 0.0
# CIRCLE_RADIUS_FRACTION = 0.25
# CIRCLE_WIDTH = 2.1
# CIRCLE_AMPLITUDE = 0.95
# CASE_NAMES['M00'] = ('primary',)  # optionally add 'passive' for lambda0=0

if not SELECTED_MODELS or not SEEDS or len(set(SEEDS)) != len(SEEDS):
    raise ValueError('Select at least one model and a nonempty list of distinct seeds.')
if len(set(SELECTED_MODELS)) != len(SELECTED_MODELS):
    raise ValueError('SELECTED_MODELS must not contain duplicates.')
if not DT_FACTORS or any(not np.isfinite(factor) or factor <= 0 for factor in DT_FACTORS):
    raise ValueError('DT_FACTORS must contain positive finite factors.')
if len(set(DT_FACTORS)) != len(DT_FACTORS):
    raise ValueError('DT_FACTORS must not contain duplicates.')
if not isinstance(SIMULATION_BATCH_SIZE, int) or isinstance(SIMULATION_BATCH_SIZE, bool) or SIMULATION_BATCH_SIZE < 1:
    raise ValueError('SIMULATION_BATCH_SIZE must be a positive integer.')

device = resolve_simulation_device(SIMULATION_DEVICE)
available_cases = {mid: control_configs(mid, n=N, **MODEL_OVERRIDES[mid]) for mid in SELECTED_MODELS}
display({mid: list(cases) for mid, cases in available_cases.items()})

def scaled_steps(base_steps, factor):
    value = base_steps / factor
    rounded = round(value)
    if not np.isclose(value, rounded, atol=1e-9, rtol=0):
        raise ValueError('Selected dt cannot represent the same physical time with integer steps.')
    return int(rounded)

groups, plan_rows = [], []
for mid, cases in available_cases.items():
    names = CASE_NAMES[mid]
    if not names or len(set(names)) != len(names):
        raise ValueError(f'{mid}: choose nonempty distinct case names.')
    for case in names:
        if case not in cases:
            raise ValueError(f'{mid}: unknown case {case}; choose {list(cases)}')
        for factor in DT_FACTORS:
            run_config = RunConfig(
                dt=DT * factor, burn_in_steps=scaled_steps(BURN_IN_STEPS, factor),
                measurement_steps=scaled_steps(MEASUREMENT_STEPS, factor),
                block_steps=scaled_steps(BLOCK_STEPS, factor),
                observation_stride=scaled_steps(OBSERVATION_STRIDE, factor),
                observation_lags=tuple(scaled_steps(lag, factor) for lag in OBSERVATION_LAGS),
                max_distance=MAX_DISTANCE,
                initial_mode=INITIAL_MODE, initial_scale=INITIAL_NOISE_SCALE,
                circle_radius_fraction=CIRCLE_RADIUS_FRACTION,
                circle_width=CIRCLE_WIDTH, circle_amplitude=CIRCLE_AMPLITUDE,
            )
            configs = [replace(run_config, seed=seed) for seed in SEEDS]
            tags = [f'{mid}_{case}_dt{run_config.dt:g}_seed{seed}' for seed in SEEDS]
            groups.append({'model': cases[case], 'configs': configs, 'tags': tags})
            for config, tag in zip(configs, tags):
                plan_rows.append({'run_id': tag, 'model_id': mid, 'case': case, 'seed': config.seed,
                    'dt': config.dt, 'burn_in_steps': config.burn_in_steps,
                    'measurement_steps': config.measurement_steps,
                    'burn_in_time': config.burn_in_steps * config.dt,
                    'measurement_time': config.measurement_steps * config.dt,
                    'observation_interval': config.observation_stride * config.dt,
                    'frames': config.measurement_steps // config.observation_stride + 1})
plan = pd.DataFrame(plan_rows)
display(plan)
print('Selected simulation backend:', device)
print('Maximum parallel replicas:', min(SIMULATION_BATCH_SIZE, len(SEEDS)) if device.startswith('cuda') else 1)
print('Total integration steps:', f"{sum(row['burn_in_steps'] + row['measurement_steps'] for row in plan_rows):,}")
print('Estimated field storage:', f"{sum(row['frames'] for row in plan_rows) * N * N * 8 / 2**30:.3f} GiB", '(plus diagnostics)')

stamp = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d-%H%M%S-%f')
OUTPUT = ROOT / 'results' / 'amb_kernel' / 'sanity' / stamp
OUTPUT.mkdir(parents=True, exist_ok=False)
plan.to_csv(OUTPUT / 'run_plan.csv', index=False)
(OUTPUT / 'settings.json').write_text(json.dumps({
    'selected_models': SELECTED_MODELS, 'case_names': CASE_NAMES, 'physical_overrides': MODEL_OVERRIDES,
    'seeds': SEEDS, 'N': N, 'base_dt': DT, 'dt_factors': DT_FACTORS,
    'simulation_device_requested': SIMULATION_DEVICE, 'simulation_device': device,
    'simulation_batch_size': SIMULATION_BATCH_SIZE,
    'video': {'run_path': str(VIDEO_RUN_PATH) if VIDEO_RUN_PATH is not None else None,
              'frame_stride': VIDEO_FRAME_STRIDE, 'max_frames': VIDEO_MAX_FRAMES, 'fps': VIDEO_FPS},
    'groups': [{'model': asdict(group['model']), 'runs': [asdict(config) for config in group['configs']]}
               for group in groups],
}, indent=2), encoding='utf8')
print('Output:', OUTPUT)

### 초기 profile 확인 · t=0, burn-in 이전

선택한 model/control과 첫 seed의 실제 초기조건 생성 함수를 호출한 field입니다.
Circle은 periodic distance와 tanh 경계를 사용하며 평균을 phi_bar에 맞춥니다.
Radius fraction, 경계 width, amplitude, 초기 noise를 설정에서 조절합니다.
물리 a/b 등의 계수는 바꾸지 않으며 circle이 burn-in 동안 유지될지는 선택한 dynamics에 달려 있습니다.
이 preview와 뒤의 burn-in 이후 field profile을 구분해 확인하세요.

**중앙 원형 클러스터 후보:** 설정 cell의 주석 처리된 M00 block을 함께 활성화하세요.
기존 a=+1 대신 a=-1, b=1을 사용하여 두 상의 공존을 허용하고, D=0.01로 noise를 낮춥니다.
약한 비국소 g1=0.1에 대한 bulk 근사 진폭은 sqrt(-(a+g1)/b)≈0.95입니다.
R/N=0.25의 원 면적비는 약 0.196이므로 평균 밀도는 (2×0.196−1)×0.95≈−0.58에서 시작합니다.
유한 경계 두께를 고려한 시작값으로 phi_bar=-0.57, width=2.1을 제안합니다.
이는 약한 activity의 탐색 시작값이며 정확한 coexistence나 장시간 안정성을 보장하지 않습니다.
lambda0=0인 passive control과 비교할 수 있습니다. 주기 경계에는 중심 고정력이 없어 클러스터가 이동할 수 있습니다.
Activity가 coexistence를 바꾸는 근거: [Wittkowski et al., Active Model B](https://www.nature.com/articles/ncomms5351).

In [ ]:
initial_profiles = {}
for mid in SELECTED_MODELS:
    for case in CASE_NAMES[mid]:
        model = AMBKernelModel(available_cases[mid][case])
        initial_profiles[(mid, case)] = model.initial_field(
            np.random.default_rng(SEEDS[0]), scale=INITIAL_NOISE_SCALE, mode=INITIAL_MODE,
            circle_radius_fraction=CIRCLE_RADIUS_FRACTION, circle_width=CIRCLE_WIDTH,
            circle_amplitude=CIRCLE_AMPLITUDE)
columns = min(3, len(initial_profiles))
rows = int(np.ceil(len(initial_profiles) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.8 * rows), squeeze=False)
vmin = min(float(field.min()) for field in initial_profiles.values())
vmax = max(float(field.max()) for field in initial_profiles.values())
for ax, ((mid, case), field) in zip(axes.flat, initial_profiles.items()):
    image = ax.imshow(field, origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set(title=f'{mid}/{case}: t=0 before burn-in', xlabel='x', ylabel='y')
    fig.colorbar(image, ax=ax, shrink=.75)
    print(f'{mid}/{case}: initial mean={field.mean():.8g}, shape={field.shape}')
for ax in list(axes.flat)[len(initial_profiles):]:
    ax.axis('off')
fig.tight_layout()
plt.show()

## 2. Kernel와 균일 상태의 선형 안정성

중앙의 밝은 점은 초기 입자나 클러스터가 아니라 **상호작용 kernel K**의 원점입니다.
한 점에 단위 입력을 주었을 때 convolution이 주변으로 전달하는 공간적 가중치, 즉 impulse response를 그린 것입니다.
이 cell은 점을 놓고 시간 적분하는 추가 시뮬레이션을 실행하지 않습니다.
K는 영향 범위·비등방성을, odd K는 방향을 뒤집었을 때 달라지는 성분을 보여 줍니다.
M00/M01의 대칭 kernel에서는 odd K가 0에 가깝습니다.

Kernel 원점은 FFT index `(0,0)`이고 그림에만 `fftshift`를 적용합니다. 양의 stiffness는 균일 상태의 선형 안정성 진단이며 nonlinear stationarity나 비가역성을 증명하지 않습니다. M01/M02의 Gaussian linearization은 reciprocal일 수 있습니다. Odd kernel은 shared color scale로 비교하여 roundoff를 큰 신호처럼 표시하지 않습니다.

In [ ]:
case_models = {(mid, name): AMBKernelModel(available_cases[mid][name])
               for mid in SELECTED_MODELS for name in CASE_NAMES[mid]}
kernel_rows = []
for (mid, name), model in case_models.items():
    active_modes = model.Lhat > 0
    kernel_rows.append({'model_id': mid, 'case': name, 'kernel_sum': float(model.K.sum()),
        'kernel_min': float(model.K.min()), 'odd_kernel_L1': float(np.abs(model.Ko).sum()),
        'shell_partition_max_error': float(np.max(np.abs(model.K_shells.sum(axis=0) - model.K))),
        'kernel_tail_L1': float(np.abs(model.K)[model.shell_labels > MAX_DISTANCE].sum() / np.abs(model.K).sum()),
        'min_real_linear_stiffness': float(np.min(model.linear_symbol().real[active_modes]))})
kernel_table = pd.DataFrame(kernel_rows)
display(kernel_table)
kernel_table.to_csv(OUTPUT / 'kernel_checks.csv', index=False)

odd_limit = max(1e-12, max(float(np.abs(model.Ko).max()) for model in case_models.values()))
fig, axes = plt.subplots(len(case_models), 3, figsize=(12, 3.5 * len(case_models)), squeeze=False)
for row, ((mid, name), model) in enumerate(case_models.items()):
    for col, values, title in ((0, np.fft.fftshift(model.K), 'K: point-source response'), (1, np.fft.fftshift(model.Ko), 'odd K: directional part')):
        image = axes[row, col].imshow(values, origin='lower', cmap='RdBu_r' if col else 'viridis',
                                     **(dict(vmin=-odd_limit, vmax=odd_limit) if col else {}))
        axes[row, col].set_title(f'{mid}/{name}: {title}')
        fig.colorbar(image, ax=axes[row, col], shrink=.75)
    offsets = np.arange(N) - N // 2
    shifted = np.fft.fftshift(model.K)
    axes[row, 2].plot(offsets, shifted[N // 2], label='x-axis')
    axes[row, 2].plot(offsets, shifted[:, N // 2], label='y-axis')
    axes[row, 2].set(title='Kernel axis profiles', xlabel='signed offset')
    axes[row, 2].legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'kernels_and_profiles.png', dpi=140)
plt.show()

## 3. 선택한 run 실행

CUDA batch는 같은 model/control/dt의 독립 seeds로 구성합니다. Batch 크기는 sampling 조건을 바꾸지 않습니다. 각 run의 EPR는 모든 integration step에서 누적하며, field는 설정한 observation interval로 저장합니다. Solver 실패 시 partial 결과를 보존하고 중단합니다. Noise 재추출이나 clipping으로 실패를 숨기지 않습니다.

In [ ]:
simulation_started = perf_counter()
run_paths = {}
work = []
for group in groups:
    width = SIMULATION_BATCH_SIZE if device.startswith('cuda') else 1
    for start in range(0, len(group['configs']), width):
        work.append((group['model'], group['configs'][start:start + width], group['tags'][start:start + width]))

for model_config, configs, tags in tqdm(work, desc='Selected physical runs', unit='batch',
                                       disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    paths = [OUTPUT / 'runs' / tag for tag in tags]
    if device.startswith('cuda'):
        from data.amb_kernel.torch_experiment import run_experiments_batched
        run_experiments_batched(model_config, configs, paths, device=device, show_progress=SHOW_PROGRESS)
    else:
        result = run_experiment(model_config, configs[0], output_dir=paths[0],
                                show_progress=SHOW_PROGRESS, progress_desc=tags[0])
        del result
    run_paths.update(dict(zip(tags, paths)))

simulation_seconds = perf_counter() - simulation_started
print('Simulation elapsed:', timedelta(seconds=round(simulation_seconds)))
(OUTPUT / 'run_paths.json').write_text(json.dumps({tag: str(path) for tag, path in run_paths.items()}, indent=2), encoding='utf8')

### 선택한 run의 profile 영상

저장된 burn-in 이후 측정 구간을 재생합니다. 초기 t=0 그림은 앞의 preview이며, 저장하지 않은 burn-in 과정은 영상에 넣지 않습니다.
컬러 범위는 영상 전체에 고정하고 제목에는 실제 물리 시간을 표시합니다.
VIDEO_FRAME_STRIDE와 VIDEO_MAX_FRAMES는 표시할 frame 수만 줄이며 첫·마지막 frame을 포함합니다.
VIDEO_FPS는 재생 속도입니다. HTML로 저장하므로 ffmpeg 없이 재생할 수 있습니다.

새 run은 RUN_TO_PLOT(미지정이면 첫 run)을 사용합니다. **기존 결과를 다시 볼 때는 import cell만 실행한 뒤**,
아래 cell 맨 위의 VIDEO_RUN_PATH에 `.../runs/<run_id>` 경로를 지정하고 이 cell을 실행하세요.
시뮬레이션을 다시 실행하지 않으며 영상은 해당 run 폴더의 profile_video.html에 저장합니다.

In [ ]:
# Existing result example: VIDEO_RUN_PATH = r'C:/.../runs/M00_primary_dt0.01_seed73031'
# Optional replay overrides: VIDEO_FRAME_STRIDE = 5; VIDEO_MAX_FRAMES = 200; VIDEO_FPS = 10
video_source = globals().get('VIDEO_RUN_PATH')
if video_source is None:
    if not globals().get('run_paths'):
        raise ValueError('Run the simulation first, or set VIDEO_RUN_PATH to an existing run folder.')
    video_tag = globals().get('RUN_TO_PLOT') or next(iter(run_paths))
    if video_tag not in run_paths:
        raise ValueError('RUN_TO_PLOT must be a saved run_id.')
    video_source = run_paths[video_tag]
video_source = Path(video_source)
video_result = load_experiment(video_source, mmap_mode='r')
try:
    video_export = render_profile_video(
        video_result.fields, video_result.times, video_source / 'profile_video.html',
        title=video_source.name, fps=globals().get('VIDEO_FPS', 10),
        frame_stride=globals().get('VIDEO_FRAME_STRIDE', 5),
        max_frames=globals().get('VIDEO_MAX_FRAMES', 200), show_progress=SHOW_PROGRESS)
finally:
    close_fields(video_result)
print('Video:', video_export['path'], '| frames:', len(video_export['frame_indices']))
display(HTML(video_export['path'].read_text(encoding='utf8')))

## 4. Mass / solver / stationarity / physical EPR

표의 block SE는 block 길이가 correlation time보다 충분히 긴 경우에 해석합니다. `stationarity_status`가 충분하더라도 seed/dt/lag 비교와 느린 morphology mode를 별도로 검토해야 합니다. 개별 channel과 tail의 signed 기여를 유지하며 total과 중복 합산하지 않습니다.

In [ ]:
physics_rows, diagnostic_rows, agreement_rows, series = [], [], [], {}
plan_by_id = {row['run_id']: row for row in plan_rows}
for tag, path in tqdm(list(run_paths.items()), desc='Read physical diagnostics', unit='run',
                      disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    result = load_experiment(path, mmap_mode='r')
    try:
        context = {key: plan_by_id[tag][key] for key in ('run_id', 'model_id', 'case', 'seed', 'dt')}
        physics_rows.extend({**context, **row} for row in result.ep_blocks)
        run_cfg = result.manifest['run_config']
        agreement_rows.append({**context,
            'burn_in_time': run_cfg['burn_in_steps'] * run_cfg['dt'],
            **compare_epr_methods(result.block_values, result.block_step_counts,
                                  dt=run_cfg['dt'], full_block_steps=run_cfg['block_steps'])})
        d = result.diagnostics
        diagnostic_rows.append({**context,
            'mass_maximum_absolute_drift': d['mass']['maximum_absolute_drift'],
            'solver_maximum_residual_norm': d['solver']['maximum_residual_norm'],
            'solver_maximum_iterations': d['solver']['maximum_iterations'],
            'stationarity_status': d['stationarity']['status'],
            'sampling_checks_passed': d['stationarity']['sampling_checks_passed'],
            'slowest_tau_int': d['stationarity']['slowest_measured_tau_int'],
            'kernel_tail_L1': d['tail']['kernel_tail_L1'],
            'elapsed_seconds': result.manifest.get('elapsed_seconds')})
        series[tag] = {'observables': result.observables, 'blocks': result.block_values,
                       'block_counts': result.block_step_counts}
    finally:
        close_fields(result)

physics = pd.DataFrame(physics_rows)
diagnostics = pd.DataFrame(diagnostic_rows)
total_physics = physics[(physics.mechanism == 'total') & (physics.shell == 'all')].copy()
display(diagnostics)
display(total_physics)
method_agreement = pd.DataFrame(agreement_rows)
display(method_agreement)
method_agreement.to_csv(OUTPUT / 'paired_method_comparison.csv', index=False)
diagnostics.to_csv(OUTPUT / 'diagnostics_summary.csv', index=False)
physics.to_csv(OUTPUT / 'physical_epr.csv', index=False)

chosen = RUN_TO_PLOT or next(iter(run_paths))
if chosen not in run_paths:
    raise ValueError('RUN_TO_PLOT must be one of the displayed run_id values.')
selected = load_experiment(run_paths[chosen], mmap_mode='r')
try:
    first, last = np.array(selected.fields[0, 0]), np.array(selected.fields[-1, 0])
finally:
    close_fields(selected)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
field_limit = max(float(np.abs(first).max()), float(np.abs(last).max()), 1e-12)
for ax, field, title in zip(axes[:2], (first, last), ('After burn-in', 'Final field')):
    image = ax.imshow(field, origin='lower', cmap='RdBu_r', vmin=-field_limit, vmax=field_limit)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, shrink=.75)
axes[2].plot(first[N // 2], label='first stored x cut (after burn-in)')
axes[2].plot(last[N // 2], label='final x cut')
axes[2].plot(last[:, N // 2], label='final y cut')
axes[2].set(title='Scalar field profiles', xlabel='lattice index')
axes[2].legend()
fig.suptitle(chosen)
plt.tight_layout()
plt.savefig(OUTPUT / 'field_profiles.png', dpi=140)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for tag, values in series.items():
    obs = values['observables']
    for ax, key in zip(axes.flat[:3], ('phi2', 'gradient_energy_density', 'low_q_structure_factor')):
        ax.plot(obs['times'], obs[key], label=tag)
        ax.set(title=key, xlabel='physical time')
    times = plan_by_id[tag]['burn_in_time'] + np.cumsum(values['block_counts']) * plan_by_id[tag]['dt']
    axes[1, 1].plot(times, values['blocks']['midpoint|total|all'], marker='.', label=tag)
axes[1, 1].set(title='Physical midpoint EPR block mean', xlabel='physical time')
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT / 'stationarity_observables.png', dpi=140)
plt.show()

## 5. Signed shell 기여와 seed / dt 비교

Shell 그림은 선택한 한 run의 active mechanism만 보여 줍니다. `r<=R`과 `tail>R`이 겹치지 않도록 표시합니다. Seed 표의 SE는 독립 run 평균 사이의 변동이며 run 내부 block SE와 다른 열입니다. Seed가 하나이면 seed SE는 정의하지 않습니다. 이 경우 그래프는 run 내부 block SE를 **잠정 오차**로 표시합니다. 계산 불가능한 오차는 0으로 대체하지 않습니다. dt 비교는 위에서 맞춘 동일한 물리 시간 구간을 사용합니다.

앞의 paired_method_comparison은 동일 trajectory의 midpoint−drift 차이를 **같은 block끼리 뺀 뒤** 오차를 구합니다. 두 추정치는 상관되므로 각 SE를 독립이라고 가정해 합치지 않습니다. gap_correlation_resolved=False이면 block 간 상관이 해소됐는지 확인하지 못한 것이므로 두 값의 일치/불일치를 확정하지 않습니다. 음수의 유한 표본 active work를 0으로 자르지 않습니다.

In [ ]:
row = plan_by_id[chosen]
physical_model = case_models[(row['model_id'], row['case'])]
mechanisms = {key.split(':')[0] for key in physical_model.active_channels(last)}
shells = physics[(physics.run_id == chosen) & (physics.method == 'midpoint')
                 & physics.mechanism.isin(mechanisms)].copy()
shells = shells[shells.shell.map(lambda label: label == 'tail' or
                                (str(label).startswith('r') and int(label[1:]) <= MAX_DISTANCE))]
labels = sorted(set(shells.shell), key=lambda label: float('inf') if label == 'tail' else int(label[1:]))
positions = {label: i for i, label in enumerate(labels)}
fig, ax = plt.subplots(figsize=(9, 4))
for mechanism, group in shells.groupby('mechanism'):
    group = group.assign(position=group.shell.map(positions)).sort_values('position')
    line, = ax.plot(group.position, group['mean'], marker='o', label=mechanism)
    measured = group[np.isfinite(group.SE)]
    ax.errorbar(measured.position, measured['mean'], yerr=measured.SE,
                fmt='none', capsize=3, color=line.get_color())
ax.axhline(0, color='black', lw=.7)
ax.set_xticks(range(len(labels)), [f'tail>{MAX_DISTANCE}' if value == 'tail' else value for value in labels])
ax.set(title=chosen + ' (bars: provisional block SE)', xlabel='Chebyshev shell', ylabel='signed extensive EP rate')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'physical_shell_epr.png', dpi=140)
plt.show()

seed_summary = total_physics.groupby(['model_id', 'case', 'dt', 'method'])['mean'].agg(
    seed_mean='mean', seed_std='std', n_seeds='count').reset_index()
seed_summary['seed_SE'] = seed_summary.seed_std / np.sqrt(seed_summary.n_seeds)
block_summary = total_physics.groupby(['model_id', 'case', 'dt', 'method']).agg(
    single_run_block_SE=('SE', lambda values: values.iloc[0] if len(values) == 1 else np.nan),
    min_full_blocks=('n_full_blocks', 'min'), measurement_time_min=('duration', 'min'),
    measurement_time_max=('duration', 'max')).reset_index()
seed_summary = seed_summary.merge(block_summary, on=['model_id', 'case', 'dt', 'method'])
seed_summary['display_SE'] = seed_summary.seed_SE.where(
    seed_summary.n_seeds > 1, seed_summary.single_run_block_SE)
seed_summary['uncertainty_source'] = np.where(seed_summary.n_seeds > 1,
    'between-seed SE', 'within-run block SE (provisional)')
seed_summary.loc[~np.isfinite(seed_summary.display_SE), 'uncertainty_source'] = 'unavailable'
display(seed_summary)
seed_summary.to_csv(OUTPUT / 'seed_dt_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4))
for key, group in seed_summary.groupby(['model_id', 'case', 'method']):
    group = group.sort_values('dt')
    line, = ax.plot(group.dt, group.seed_mean, marker='o', label='/'.join(key))
    measured = group[np.isfinite(group.display_SE)]
    ax.errorbar(measured.dt, measured.seed_mean, yerr=measured.display_SE,
                fmt='none', capsize=3, color=line.get_color())
    for point in group.itertuples():
        if not np.isfinite(point.display_SE):
            ax.annotate('SE unavailable', (point.dt, point.seed_mean), xytext=(5, 5),
                        textcoords='offset points', fontsize=8)
ax.axhline(0, color='black', lw=.7)
ax.set(title='Bars: 1 SE; single-seed bars use provisional within-run blocks',
       xlabel='integration dt', ylabel='extensive EP rate', xscale='log')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT / 'seed_dt_epr.png', dpi=140)
plt.show()

## 6. Finite observation lag

Endpoint midpoint work는 관측 pair에서 계산하는 유한 lag quantity입니다. Integration-step streaming EPR와 같다고 간주하지 않습니다. 같은 physical lag의 dt별 값을 비교할 수 있도록 각 행에 observation interval을 기록합니다. 아래 표는 nonoverlapping observation intervals의 active work를 합산합니다.

In [ ]:
lag_rows = []
for tag, path in tqdm(list(run_paths.items()), desc='Finite-lag endpoint work', unit='run',
                      disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    result = load_experiment(path, mmap_mode='r')
    try:
        cfg = result.manifest['run_config']
        frame_dt = cfg['dt'] * cfg['observation_stride']
        for lag_steps in cfg['observation_lags']:
            lag = lag_steps // cfg['observation_stride']
            works = []
            for start in range(0, len(result.fields) - lag, lag):
                left, right = result.fields[start, 0], result.fields[start + lag, 0]
                active = result.model.active_mu(.5 * (left + right))
                works.append(-float(np.sum(active * (right - left))) / result.model.config.D)
            lag_rows.append({'run_id': tag, 'model_id': plan_by_id[tag]['model_id'],
                'case': plan_by_id[tag]['case'], 'seed': cfg['seed'], 'dt': cfg['dt'],
                'physical_lag': lag * frame_dt, 'n_intervals': len(works),
                'endpoint_rate': float(np.mean(works)) / (lag * frame_dt) if works else None})
    finally:
        close_fields(result)
lag_table = pd.DataFrame(lag_rows)
display(lag_table)
lag_table.to_csv(OUTPUT / 'finite_lag_work.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 4))
for tag, group in lag_table.groupby('run_id'):
    ax.plot(group.physical_lag, group.endpoint_rate, marker='o', label=tag)
ax.axhline(0, color='black', lw=.7)
ax.set(xlabel='physical observation lag', ylabel='endpoint active-work rate', xscale='log')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT / 'finite_lag_work.png', dpi=140)
plt.show()

## 7. M03 Gaussian calibration

M03의 `b=0` run을 선택하면 analytic continuous-time covariance/EPR와 비교합니다. `abs(FFT(phi-mean))²/N`이 covariance eigenvalue와 같은 정규화입니다. 유한 표본과 finite-dt bias를 분리하여 해석하세요. 이 cell은 선택하지 않은 추가 trajectory를 생성하지 않습니다.

In [ ]:
gaussian_rows = []
for tag, path in run_paths.items():
    info = plan_by_id[tag]
    if info['model_id'] != 'M03' or case_models[(info['model_id'], info['case'])].config.b != 0:
        continue
    result = load_experiment(path, mmap_mode='r')
    try:
        reference = result.model.gaussian_reference(max_distance=MAX_DISTANCE)
        empirical = np.zeros(result.model.shape)
        for start in range(0, len(result.fields), 64):
            fields = np.asarray(result.fields[start:start + 64, 0], dtype=np.float64)
            centered = fields - fields.mean(axis=(-2, -1), keepdims=True)
            empirical += (np.abs(np.fft.fft2(centered)) ** 2 / result.model.N).sum(axis=0)
        empirical /= len(result.fields)
        covariance = reference['covariance_spectrum']
        error = float(np.linalg.norm(empirical - covariance) / np.linalg.norm(covariance))
        midpoint = next(row['mean'] for row in result.ep_blocks
                        if row['method'] == 'midpoint' and row['mechanism'] == 'total' and row['shell'] == 'all')
        gaussian_rows.append({'run_id': tag, 'sigma_reference': reference['sigma_total'],
                              'sigma_midpoint': midpoint, 'covariance_relative_error': error,
                              'min_H': reference['min_H'], 'n_frames': len(result.fields)})
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, values, title in zip(axes, (covariance, empirical), ('Analytic covariance spectrum', 'Empirical spectrum')):
            image = ax.imshow(np.fft.fftshift(values), origin='lower')
            ax.set_title(title)
            fig.colorbar(image, ax=ax, shrink=.75)
        fig.suptitle(tag)
        plt.tight_layout()
        plt.savefig(OUTPUT / f'{tag}_gaussian_covariance.png', dpi=140)
        plt.show()
    finally:
        close_fields(result)
if gaussian_rows:
    gaussian_table = pd.DataFrame(gaussian_rows)
    display(gaussian_table)
    gaussian_table.to_csv(OUTPUT / 'gaussian_calibration.csv', index=False)
else:
    print('Select M03 with b=0 in the settings cell to run Gaussian covariance/EPR calibration.')

elapsed = perf_counter() - STARTED
(OUTPUT / 'elapsed_summary.json').write_text(json.dumps({
    'simulation_seconds': simulation_seconds, 'notebook_wall_seconds': elapsed,
    'run_count': len(run_paths), 'simulation_device': device,
}, indent=2), encoding='utf8')
print('Total elapsed:', timedelta(seconds=round(elapsed)), '| results:', OUTPUT)

## 판정과 다음 단계

Mass/solver/kernel 오류가 없는지 확인하고, independent seeds와 충분한 stationary blocks, dt/lag 수렴을 검토합니다. `sigma > 3*block_SE` 하나만으로 물리 검증을 완료하지 않습니다. Reciprocal/null controls와 Gaussian calibration은 해당 모델의 조건에서 비교하며 M05의 `u*u_r`는 attribution convention입니다.

Parameter를 결정한 뒤 학습 notebook의 `MODEL_OVERRIDES`, `TRAIN_DT`, burn-in/interval 설정에 반영합니다. 학습 notebook은 독립 train/validation/test ensemble과 NN 학습·평가만 수행합니다. 모든 원본 정의는 [공통 명세](../data/amb_kernel/specs/COMMON_PROTOCOL.md)와 [모델 목록](../data/amb_kernel/specs/README.md)에 있습니다.